# Replogle K562: compact four-panel figure

This notebook reads saved results only.
Run `python -m experiments.replogle.run` first to reproduce the analysis.
Panels (a) and (b) compare pair-level RMSE against the all-data raw-difference reference, which is an empirical reference rather than causal ground truth.
Panels (c) and (d) show the same fixed gene panel and perturbation order.
RCB effects are differences of mean log1p expression divided by ln(2), as in the original figure notebook; Wilcoxon values are the supplied log2 fold changes, so the quantities are distinct.
Stars in (c) mark shifted model-based prediction intervals excluding zero, not ATT confidence intervals; stars in (d) mark adjusted Wilcoxon p-values below 0.05.
The color range is fixed at -3 to 3, with out-of-range values clipped.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
from rcb import paths

OUTPUT = paths.results("replogle", create=False)
FIGURES = paths.figures("replogle")
manifest_path = OUTPUT / "run_manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    if manifest["status"] != "complete":
        raise RuntimeError("Replogle run is incomplete; regenerate results before plotting")
    print("Results mode:", manifest["mode"])
else:
    print("Reading supplied reference results; generating configuration is not recorded.")
att_summary = pd.read_csv(OUTPUT / "att_overlap_tilt_pair_summary.csv")
full = pd.read_csv(OUTPUT / "full_k562_rcb_analysis.csv")
genes = pd.read_csv(OUTPUT / "full_k562_outcome_panel.csv")["outcome_h"].tolist()
perturbations = full["perturbation_p"].drop_duplicates().tolist()
for column in ("rcb_interval_excludes_zero", "de_significant"):
    full[column] = full[column].astype(str).str.lower().eq("true")
plot_df = att_summary.loc[att_summary["reference_type"] == "All-data raw difference"]
pairs = [
    ("Uniform", "pure_uniform", "rcb_uniform"),
    ("L2", "pure_l2", "rcb_l2"),
    ("Entropy", "pure_entropy", "rcb_entropy"),
    ("ABW\nOutcome", "double_ridge_outcome", "rcb_on_abw_outcome"),
    ("ABW\nImbalance", "double_ridge_imbalance", "rcb_on_abw_imbalance"),
    ("ABW\nRiesz", "double_ridge_riesz", "rcb_on_abw_riesz"),
]


Results mode: production


In [2]:
with plt.rc_context({"font.size": 10, "axes.titlesize": 9, "axes.labelsize": 8,
                     "xtick.labelsize": 7, "ytick.labelsize": 7, "pdf.fonttype": 42}):
    fig = plt.figure(figsize=(10, 5), layout="constrained")
    gs = fig.add_gridspec(2, 3, width_ratios=[1, 1., 1.])
    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[1, 0])
    ax_c = fig.add_subplot(gs[:, 1])
    ax_d = fig.add_subplot(gs[:, 2])

    colors = ("#9E9E9E", "#4C78A8")
    for ax, regime, panel in zip((ax_a, ax_b), ("good", "weak"), ("a", "b")):
        subset = plot_df.loc[plot_df["regime"] == regime]
        for i, (_, base_key, rcb_key) in enumerate(pairs):
            for offset, key, color, marker in zip((1, 2), (base_key, rcb_key), colors, ("s", "o")):
                values = subset.loc[subset["method_key"] == key, "rmse"].to_numpy(float)
                if not len(values) or not np.isfinite(values).all() or np.any(values <= 0):
                    raise ValueError(f"Missing or nonpositive RMSE for {regime}/{key}")
                position = 3 * i + offset
                if len(values) > 1 and np.ptp(values) > 0:
                    violin = ax.violinplot([values], positions=[position], widths=0.85,
                                           showmedians=True, showextrema=False)
                    violin["bodies"][0].set_facecolor(color)
                    violin["bodies"][0].set_edgecolor(color)
                    violin["bodies"][0].set_alpha(0.35)
                    violin["cmedians"].set_color(color)
                ax.scatter(np.full(len(values), position), values, s=2, alpha=0.3,
                    color=color, marker=marker, linewidths=0)
        ax.set_xticks([3 * i + 1.5 for i in range(len(pairs))], [p[0] for p in pairs],
                      rotation=25, ha="right")
        ax.set_ylabel("Pair RMSE")    
        ax.set_title(f"({panel}) {regime.capitalize()} overlap", loc="left")
        ax.spines[["top", "right"]].set_visible(False)
    limits = (min(ax.get_ylim()[0] for ax in (ax_a, ax_b)), 
              0.08)#max(ax.get_ylim()[1] for ax in (ax_a, ax_b)))
    for ax in (ax_a, ax_b):
        ax.set_ylim(limits)
    ax_a.legend(handles=[Patch(color=c, alpha=0.5, label=l)
                          for c, l in zip(colors, ("Base", "RCB"))],
                loc="upper right", frameon=False, ncol=2, fontsize=7,
                handlelength=1, columnspacing=0.8)

    # Panel (c): RCB contrast. Panel (d): the covariate adjustment it applies, i.e. the RCB
    # contrast minus the unadjusted contrast, on its own scale.
    full["rcb_adjustment"] = full["rcb_effect"] - full["raw_effect"]
    for ax, value, flag, limit, title in (
        (ax_c, "rcb_effect", None, 3, "(c) RCB predictive contrast"),
        (ax_d, "rcb_adjustment", None, 0.2, "(d) Covariate adjustment by RCB"),
    ):
        matrix = full.pivot(index="outcome_h", columns="perturbation_p", values=value).reindex(index=genes, columns=perturbations) / np.log(2)
        if matrix.isna().any().any():
            raise ValueError("Incomplete fixed-panel result matrix")
        im = ax.imshow(np.clip(matrix.to_numpy(float), -limit, limit), aspect="auto", cmap="RdBu_r",
                       norm=TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit))
        ax.set_xticks(range(len(perturbations)), perturbations, rotation=25, ha="right")
        ax.set_yticks(range(len(genes)), genes, fontsize=6.5)
        ax.set_xlabel("Perturbation")
        ax.set_title(title, loc="left")
        if flag is not None:
            flags = full.pivot(index="outcome_h", columns="perturbation_p", values=flag).reindex(index=genes, columns=perturbations)
            if flags.isna().any().any():
                raise ValueError("Incomplete fixed-panel flag matrix")
            for i, j in np.argwhere(flags.to_numpy(bool)):
                ax.text(j, i, "*", ha="center", va="center", fontsize=8, fontweight="bold")
        fig.colorbar(im, ax=ax, fraction=0.05, pad=0.03, ticks=[-limit, 0, limit], extend="both")
    ax_c.set_ylabel("Outcome gene")
    figure_path = FIGURES / "01_k562_overview.pdf"
    fig.savefig(figure_path, bbox_inches="tight")
    plt.show()


In [3]:
# Appendix figure: why the full-data adjustment differs across perturbations.
shift = pd.read_csv(OUTPUT / "full_k562_covariate_shift.csv")
path = pd.read_csv(OUTPUT / "full_k562_adjustment_path.csv")
colors = dict(zip(perturbations, ("#9E9E9E", "#F58518", "#BAB0AC", "#54A24B", "#4C78A8")))
qhat = full.groupby("perturbation_p")["rcb_qhat_endpoint"].first()
with plt.rc_context({"font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 7.5,
                     "pdf.fonttype": 42}):
    fig, (ax_a, ax_b, ax_c) = plt.subplots(1, 3, figsize=(7.5, 3.0), layout="constrained")

    data = [shift.loc[shift["perturbation_p"] == p, "mean_shift"].abs().to_numpy() for p in perturbations]
    box = ax_a.boxplot(data, patch_artist=True, showfliers=False, widths=0.6,
                        medianprops={"color": "black"})
    for patch, p in zip(box["boxes"], perturbations):
        patch.set_facecolor(colors[p]); patch.set_alpha(0.6)
    top = max(np.max(w.get_ydata()) for w in box["whiskers"]) * 1.05
    for i, p in enumerate(perturbations, start=1):
        ax_a.text(i, top, f"{qhat[p]:.2f}", ha="center", va="bottom", fontsize=8)
    ax_a.set_ylim(top=top * 1.1)
    ax_a.set_xticks(range(1, len(perturbations) + 1), perturbations, rotation=30, ha="right")
    ax_a.set_ylabel("|Standardised shift|")
    ax_a.set_title(r"(a) Shift and $\widehat q(\infty)$", loc="left")

    snr = full["rcb_r2"] / full["rcb_sigma2"]
    for p in perturbations:
        rows = full["perturbation_p"] == p
        finite = rows & np.isfinite(full["rcb_selected_lambda"])
        if finite.any():
            ax_b.scatter(snr[finite], full.loc[finite, "rcb_selected_lambda"], s=18,
                         color=colors[p], label=p)
        else:
            ax_b.scatter([], [], s=18, color=colors[p], label=rf"{p} ($\lambda=\infty$)")
    ax_b.set_xscale("log"); ax_b.set_yscale("log")
    ax_b.set_xlabel(r"$\widehat r^2/\widehat\sigma_0^2$")
    ax_b.set_ylabel(r"Selected penalty $\widehat\lambda$")
    ax_b.set_title("(b) Selected penalties", loc="left")
    ax_b.legend(frameon=False, loc="upper right", handletextpad=0.2, borderaxespad=0)

    finite_path = path.loc[np.isfinite(path["penalty"])].copy()
    finite_path["abs_adjustment"] = finite_path["rcb_adjustment"].abs() / np.log(2)
    for p in perturbations:
        curve = finite_path.loc[finite_path["perturbation_p"] == p].groupby("penalty")["abs_adjustment"].median()
        ax_c.plot(curve.index, curve.to_numpy(), color=colors[p], lw=1.8, label=p)
        chosen = full.loc[(full["perturbation_p"] == p) & np.isfinite(full["rcb_selected_lambda"]), "rcb_selected_lambda"]
        if len(chosen):
            lam = float(np.median(chosen))
            ax_c.scatter([lam], [float(np.interp(np.log(lam), np.log(curve.index), curve.to_numpy()))],
                         color=colors[p], s=40, zorder=3, edgecolor="black", linewidth=0.6)
    ax_c.set_xscale("log")
    ax_c.set_xlabel(r"Penalty $\lambda$")
    ax_c.set_ylabel(r"Median |adjustment|")
    ax_c.set_title("(c) Adjustment path", loc="left")
    for ax in (ax_a, ax_b, ax_c):
        ax.spines[["top", "right"]].set_visible(False)
    figure_path = FIGURES / "02_k562_adjustment_diagnostics.pdf"
    fig.savefig(figure_path, bbox_inches="tight")
    plt.show()


In [4]:
# Appendix table: balance diagnostics of the overlap-sensitivity fits (medians per regime).
raw = pd.read_csv(OUTPUT / "att_overlap_tilt_repeated_raw.csv")
balance_rows = []
for label, base_key, rcb_key in pairs:
    for weights, key in (("Base", base_key), ("RCB", rcb_key)):
        row = {"base": label.replace("\n", " "), "weights": weights}
        for regime, group in raw.groupby("regime"):
            for metric, column in (("ESS", "ess"), ("Pilot", "pilot_imbalance"),
                                   ("Population", "population_imbalance")):
                row[f"{regime} {metric}"] = group[f"{column}_{key}"].median()
        balance_rows.append(row)
print(pd.DataFrame(balance_rows).round(2).to_string(index=False))


         base weights  good ESS  good Pilot  good Population  weak ESS  weak Pilot  weak Population
      Uniform    Base     48.00       28.00            19.44     48.00       29.07            20.34
      Uniform     RCB     48.00       26.77            18.72     28.01       27.45            19.28
           L2    Base      5.05       25.62            19.28      9.67       26.90            19.65
           L2     RCB      6.18       26.44            18.72     11.63       27.30            19.44
      Entropy    Base     13.13       26.69            19.02     12.18       27.36            19.51
      Entropy     RCB     13.78       26.91            18.82     13.13       27.57            19.42
  ABW Outcome    Base      4.37       25.61            19.44      8.24       26.89            19.74
  ABW Outcome     RCB      6.66       26.53            18.76     10.63       27.33            19.46
ABW Imbalance    Base      4.46       25.61            19.37      8.27       26.89            19.74
